In [0]:
from pyspark.sql.functions import col, lag, row_number, lit, coalesce, desc
from pyspark.sql.window import Window
from project_tools import merge_into_table

In [0]:
dorm_bills = ( spark.read
              .table("workspace.project_catalog.bronze_layer")
              .drop(col("note"), col("created_at"), col("_processing_time"))
              )

dorm_rates = ( spark.read
    .table("workspace.project_catalog.dorm_rate")
    .orderBy(desc("valid_from"))
    .limit(1)
    .drop("note", "created_at", "_processing_time")
)

In [0]:
window = Window.partitionBy("room_id").orderBy("bill_date")

calculated_meter_used = ( dorm_bills
                    .withColumn("elec_unit_used", coalesce(col("elec_meter") - lag(col("elec_meter"), 1).over(window), lit(0)))
                    .withColumn("water_unit_used", coalesce(col("water_meter") - lag(col("water_meter"), 1).over(window),lit(0)))
                    .filter(col("elec_unit_used") >= 0)
                    .filter(col("water_unit_used") >= 0)
                    )

In [0]:
joined_bills = ( calculated_meter_used.alias("bill")
                        .join(dorm_rates.alias("rate"),
                               col("bill.bill_date") >= col("rate.valid_from"),
                                "left")
                        )

In [0]:
calculated_bills = (joined_bills
                    .withColumn("elec_charge", col("elec_unit_used") * col("elec_per_unit"))
                    .withColumn("water_charge", col("water_unit_used") * col("water_per_unit"))
                    .withColumn("total_charge", col("elec_charge") + col("water_charge") + col("rent"))
                    .drop(col("elec_meter"), col("water_meter"), col("rate_id"), col("valid_from"), col("elec_per_unit"), col("water_per_unit"))
                    )

In [0]:
merge_into_table(calculated_bills, ["bill_id"], "silver_layer")